# PALLAS surrogate models — kickoff notebook

**This is the long version of `kickoff_notebook.ipynb`: the same path, every step explained.** It answers three questions before you write a
line of model code — what one simulation of this machine takes **in**, what it puts **out**, and
what sits in **every file** of the pack — and then it builds two scored baselines and writes two
valid submissions. It runs in under a minute on a laptop CPU; reading it takes about forty minutes.

PALLAS is a **laser-plasma accelerator**: a laser pulse drives a plasma wave in a
few-millimetre gas cell, and electrons stripped from the nitrogen inner shell ride that wave
to between roughly 40 and 500 MeV in a very short distance. The operator sets a handful of
knobs and gets one electron bunch out. Everything here is simulation, which is deliberate —
it gives every sample an exact ground-truth label, which no measurement does.

Sections 1 to 5 are the tour: nothing is scored, each file is opened and its shape printed.
Sections 6 to 9 are the work: the columns, the two conventions, and a baseline in each
direction — the beam read back to the settings, and the settings read forward to the beam.

**`objectives.md`** lists what is actually being asked, easy to hard. **`scientific_case.md`**
explains why the machine matters. **`data/README.md`** gives the splits and the scoring rules
in full.


## Before you start

The full data pack is **not in this repository** — see [`data/README.md`](data/README.md)
for the download and unpack it as `data/pallas_hackathon_data/`. Until you have it, the
notebook falls back on a 10 % random sample in `data/sample/` (22 MB, same tables and
columns), so it runs straight from a clone; its numbers differ from the full pack's. Then:

```
pip install -r requirements.txt
```

Everything in this notebook runs on a laptop CPU in under a minute.


## The pack — find it once

Set `PALLAS_PACK` if you unpacked the data somewhere else. Every cell below reads from `PACK`,
and from the two frames this cell loads. There is no reloading later in the notebook.


In [1]:
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# The full pack is not in this folder: download it and unpack it under data/.
# Set PALLAS_PACK if you keep it anywhere else. Without it, the 10 % sample in
# data/sample/ is used, so the notebook runs straight from a clone of the repository.
CANDIDATES = [os.environ.get("PALLAS_PACK"),
              os.path.join("data", "pallas_hackathon_data"),
              os.path.join("..", "pallas_hackathon_data"),
              "pallas_hackathon_data",
              os.path.join("data", "sample")]
PACK = None
for candidate in CANDIDATES:                 # the first folder that holds the pack wins
    if candidate and os.path.exists(os.path.join(candidate, "campaign_A.parquet")):
        PACK = candidate
        break
if PACK is None:
    raise SystemExit("no data pack found -- see data/README.md for the download "
                     "and unpack it as data/pallas_hackathon_data/")

# An old or partial download stops here, not three cells later with a KeyError:
# every file and column listed in the schema must be in the pack.
with open(os.path.join("data", "ontologies", "pack_schema.json"), encoding="utf-8") as fh:
    SCHEMA = json.load(fh)
for name, spec in SCHEMA["files"].items():
    path = os.path.join(PACK, name)
    if not os.path.exists(path):
        raise SystemExit(f"{path} is missing: {PACK} is not the current pack (data/README.md)")
    if name.endswith(".parquet"):
        absent = set(spec["columns"]) - set(pq.ParquetFile(path).schema_arrow.names)
        if absent:
            raise SystemExit(f"{name} lacks {sorted(absent)}: {PACK} is not the current pack")
if PACK == os.path.join("data", "sample"):
    print("using the 10 % SAMPLE in data/sample/ -- every number below will differ from "
          "the full pack's; download the full pack before training anything (data/README.md)")

n_files = sum(1 for f in os.listdir(PACK) if os.path.isfile(os.path.join(PACK, f)))
size_mb = sum(os.path.getsize(os.path.join(PACK, f)) for f in os.listdir(PACK)
              if os.path.isfile(os.path.join(PACK, f))) / 1e6
print(f"pack at {PACK}: {n_files} files, {size_mb:.0f} MB\n")

# The array columns (spectra, density profiles) are most of the file size, so ask
# for the scalars unless you need them. Both campaigns are loaded once, here.
SCALARS = ["scan", "config", "p_1", "a_0", "c_N2", "x_of",
           "P_max", "cN2_max", "L_inj", "dip_frac",
           "E_med_MeV", "E_mean_MeV", "dE_mad", "dE_std", "q_end", "i_peak",
           "sigma_z", "sigma_y", "n_emit_x", "n_emit_y", "div_rms", "beta_x",
           "alpha_x", "z_avg", "injected"]


def load(name):
    path = f"{PACK}/{name}.parquet"
    cols = pq.ParquetFile(path).schema.names      # the footer, not the data
    return pd.read_parquet(path, columns=[c for c in SCALARS if c in cols])


# Two campaigns, two files. Campaign A holds both of its scans, `scan` says which.
A = load("campaign_A")
B = load("campaign_B")
grid = np.load(f"{PACK}/common_energy_grid_MeV.npy")   # shared energy axis, MeV


def summarise(label, frame):
    print(f"{label:22s} {len(frame):6,} rows   {frame.injected.sum():5,} injected"
          f"   ({frame.injected.mean():.1%})")


summarise("campaign_A", A)
for scan, rows in A.groupby("scan"):
    summarise(f"   scan = {scan}", rows)
summarise("campaign_B", B)


pack at data/pallas_hackathon_data: 13 files, 201 MB

campaign_A             13,073 rows   10,751 injected   (82.2%)
   scan = random        9,285 rows   7,208 injected   (77.6%)
   scan = uniform       3,788 rows   3,543 injected   (93.5%)
campaign_B              5,286 rows   2,882 injected   (54.5%)


## 1. What one simulation is

PALLAS is a laser-plasma accelerator. A high-power laser pulse is focused into a
few-millimetre cell of helium mixed with a little nitrogen. The pulse drives a wave in the
plasma behind it; electrons stripped from the inner shell of the nitrogen fall into that wave
and are accelerated to somewhere between roughly 40 and 500 MeV in a few millimetres. A
conventional accelerator needs tens of metres to do the same thing.

**One simulation is one setting of the knobs, run start to finish, producing one electron
bunch.** It is a particle-in-cell simulation run with Smilei: the code follows the laser
field, the plasma and the electrons through the cell on a grid, so the bunch that comes out
is computed from first principles rather than fitted. That is the reason for using simulation
here at all — every sample carries an exact ground-truth label, which no measurement does.

The pack holds two **campaigns**. They are two separate studies of the same machine and they
do not share a knob set, which is why they are two files.

### The inputs — everything that is chosen before a simulation runs

**Campaign A** varies four scalars: the gas, the nitrogen, and where and how hard the laser
is focused.

| input | meaning | unit |
|---|---|---|
| `p_1` | pressure in the injection region — how much gas there is | mbar |
| `a_0` | laser amplitude, the normalised vector potential — how hard the laser drives the plasma | dimensionless |
| `c_N2` | nitrogen concentration in the helium — the atoms the accelerated electrons come from | fraction |
| `x_of` | focal position — where along the cell the laser is focused | µm |

**Campaign B** fixes the laser and varies the **shape of the gas density along the cell**
instead. Its four knobs are the four numbers that describe that shape.

| input | meaning | unit |
|---|---|---|
| `P_max` | peak pressure of the density profile | Pa |
| `cN2_max` | peak nitrogen fraction | fraction |
| `L_inj` | length of the injection region | mm |
| `dip_frac` | depth of the dip in the density that triggers injection | fraction |
| `x_of` | focal position, as in Campaign A | µm |

Campaign B also ships **the density profile itself** as a 2000-point curve (`x_p`, `n_e_p`) —
the four knobs are a summary of that curve, and the curve is the thing the simulation was
actually given. Campaign A ships a profile too, but as only 6 points, because its gas cell is
described by a handful of corner positions rather than a sampled curve.

**`x_p` is in micrometres, not metres.** The cell below checks it rather than asserting it:
the curve ends exactly at `L_inj + 3.2` mm, the plasma end every trajectory score in this
pack is split at. `n_e_p` is in m⁻³.

**Held fixed in both, so it is in no column:** the cell geometry, the laser pulse duration and
wavelength, the helium, and every numerical setting of the simulation. Campaign B additionally
fixes `a_0`.

In [2]:
A_KNOBS = ["p_1", "a_0", "c_N2", "x_of"]
B_KNOBS = ["P_max", "cN2_max", "L_inj", "dip_frac", "x_of"]

cfg = int(B.config[B.injected].iloc[0])   # one simulation that produced a bunch, used all the way down
b_row = B[B.config == cfg].iloc[0]        # its row of campaign_B

print("one Campaign A simulation was asked for:")
print(A[A.injected].iloc[0][A_KNOBS].to_string(), "\n")
print(f"one Campaign B simulation was asked for (config {cfg}):")
print(b_row[B_KNOBS].to_string(), "\n")

profiles = pd.read_parquet(f"{PACK}/campaign_B.parquet", columns=["config", "x_p", "n_e_p"])
prof = profiles[profiles.config == cfg].iloc[0]
x_p = np.asarray(prof.x_p)           # position along the cell, MICROmetres
n_e_p = np.asarray(prof.n_e_p)       # electron density there, m^-3
print(f"and it was given a density curve of {x_p.size} points: "
      f"{x_p.min() / 1e3:.3f} to {x_p.max() / 1e3:.3f} mm, "
      f"peak density {n_e_p.max():.3e} m^-3")
print(f"  that curve ends at L_inj + 3.2 mm = "
      f"{b_row.L_inj + 3.2:.3f} mm, which is the plasma end this pack scores against")

one Campaign A simulation was asked for:
p_1       51.96797
a_0       1.328588
c_N2      0.031888
x_of    740.016691 

one Campaign B simulation was asked for (config 1163):
P_max       5847.603101
cN2_max         0.21759
L_inj          0.454187
dip_frac       0.266643
x_of         290.397487 



and it was given a density curve of 2000 points: 0.000 to 3.654 mm, peak density 2.638e+24 m^-3
  that curve ends at L_inj + 3.2 mm = 3.654 mm, which is the plasma end this pack scores against


### The outputs — what comes out, at four levels of detail

The same simulation is stored four times over, each time in more detail. Which one you want
decides which file you open, and it is the single most useful thing to know about this pack.

| level | what it is | where | size of one simulation |
|---|---|---|---|
| 1 | the finished bunch as **a handful of numbers** — energy, charge, spread, length, emittance, divergence | `campaign_A.parquet`, `campaign_B.parquet` | ~20 numbers |
| 2 | its **energy spectrum** — how much charge at each energy | the `spec_common` column of the same files | 200 bins |
| 3 | the bunch **along the accelerator**, not just at the end | `campaign_B_trajectories.parquet` | 113 positions × 6 numbers |
| 4 | the bunch's **full six-dimensional shape** at 20 places | `campaign_B_moments.parquet` | 20 planes × (6 means + a 6×6 covariance) |

**Levels 3 and 4 are Campaign B only.** Campaign A is stored at levels 1 and 2 and
nowhere else, so nothing in this pack says what happened *along* the accelerator for
any of its runs.

Level 4 is the richest thing here. A bunch is a cloud of electrons in six dimensions — three
positions and three momenta — and its covariance is the 6×6 table of how those six coordinates
vary together. Emittance, beam size, divergence and the energy chirp are all things you can
compute *from* that table; none of them contains it.

**Levels 3 and 4 trade against each other, and they do not cover the same simulations.** Level 3
samples the accelerator finely — 113 positions — but keeps only six derived numbers at each,
so every correlation between coordinates is gone. Level 4 samples it coarsely — 20 planes — and
keeps everything second-order, including the chirp and the position–divergence correlation that
level 3 discards. And level 3 covers **2,855 configurations against level 4's 5,286**, so the
trajectory table is about half the corpus. The cell below prints both counts.

In [3]:
# Level 1 -- the bunch as a few numbers. The units are the trap, so they are printed.
scal = pd.read_parquet(f"{PACK}/campaign_B.parquet",
                       columns=["config", "E_med_MeV", "dE_mad", "q_pC", "q_end", "i_peak",
                                "sigma_z", "n_emit_x", "div_rms"])
r = scal[scal.config == cfg].iloc[0]
print(f"level 1 -- config {cfg} as numbers, each with the unit it is actually in:")
for col, unit, note in [
        ("E_med_MeV", "MeV", "median energy of the bunch"),
        ("dE_mad", "fraction", "energy spread -- a FRACTION, not MeV"),
        ("q_pC", "pC", "bunch charge"),
        ("q_end", "C", "the same charge in coulombs, so 3 pC is 3e-12"),
        ("i_peak", "A", "peak current"),
        ("sigma_z", "m", "bunch length, RMS"),
        ("n_emit_x", "m.rad", "normalised emittance -- how tightly focusable the bunch is"),
        ("div_rms", "rad", "divergence, RMS")]:
    print(f"  {col:12s} {r[col]:>14.6g}  {unit:<9s} {note}")
print()

# Level 2 -- the spectrum, on the grid shared by every row of both campaigns.
sp = pd.read_parquet(f"{PACK}/campaign_B.parquet", columns=["config", "spec_common"])
s = np.asarray(sp[sp.config == cfg].iloc[0].spec_common)   # pC in each shared bin
print(f"level 2 -- the spectrum: {s.size} bins over "
      f"{grid[0]:.1f}-{grid[-1]:.1f} MeV, peak at {grid[s.argmax()]:.1f} MeV\n")

level 1 -- config 1163 as numbers, each with the unit it is actually in:
  E_med_MeV           113.582  MeV       median energy of the bunch
  dE_mad             0.102667  fraction  energy spread -- a FRACTION, not MeV
  q_pC                 115.26  pC        bunch charge
  q_end            1.1526e-10  C         the same charge in coulombs, so 3 pC is 3e-12
  i_peak              5834.84  A         peak current
  sigma_z         2.41426e-06  m         bunch length, RMS
  n_emit_x        1.10348e-06  m.rad     normalised emittance -- how tightly focusable the bunch is
  div_rms          0.00100617  rad       divergence, RMS

level 2 -- the spectrum: 200 bins over 25.8-510.3 MeV, peak at 106.1 MeV



In [4]:
# Level 3 -- the beam at 113 positions along the accelerator.
traj = pd.read_parquet(f"{PACK}/campaign_B_trajectories.parquet")
t = traj[traj.config == cfg].sort_values("z_mm")
print(f"level 3 -- {len(t)} positions from {t.z_mm.min():.3f} to {t.z_mm.max():.3f} mm, "
      f"for {traj.config.nunique():,} of the pack's configurations:")
print(t[["z_mm", "E_MeV", "q_pC", "dE_pct", "emit_um", "div_mrad", "sigz_um"]]
      .head(3).to_string(index=False), "\n")

# Level 4 -- the 6-D bunch at 20 planes. `sim_id` is the odd one out; see section 3.
# `group` splits every plane in two: "all" is every stored macroparticle, "cohort" a
# selected sub-population. Pick one and stay with it -- "all" is used throughout this pack,
# and it is the group the hidden truth is built from.
mom = pd.read_parquet(f"{PACK}/campaign_B_moments.parquet")
mom["config"] = mom.sim_id.str.removeprefix("Config_").astype(int)
m = mom[(mom.config == cfg) & (mom.group == "all")].sort_values("plane_mm")
cov_cols = [c for c in mom.columns if c.startswith("cov_")]
print(f"level 4 -- {len(m)} planes from {m.plane_mm.min():.2f} to {m.plane_mm.max():.2f} mm, "
      f"6 means and {len(cov_cols)} covariance entries each, "
      f"for {mom[mom.group == 'all'].config.nunique():,} configurations "
      f"(group values: {', '.join(sorted(mom.group.unique()))})")
print("  the 6 coordinates are (zeta, y, z, ux, uy, uz):",
      "zeta along the bunch in um, y and z across it in um, and the three momenta")
row = m.iloc[len(m) // 2]
C = np.zeros((6, 6))
for i in range(6):
    for j in range(i, 6):
        C[i, j] = C[j, i] = row[f"cov_{i}{j}"]
print(f"\n  the covariance at plane {row.plane_mm:.2f} mm, as the 6x6 matrix it is:")
print(np.array2string(C, precision=3, suppress_small=False))

level 3 -- 113 positions from 2.015 to 7.054 mm, for 2,855 of the pack's configurations:
    z_mm     E_MeV       q_pC   dE_pct   emit_um  div_mrad  sigz_um
2.015055 42.049046 133.855072 8.952054 11.427250 26.040369 3.539988
2.060053 45.188580 133.854492 8.831645 12.880788 24.394506 3.607092
2.105055 48.405617 133.854477 8.774195 14.642128 22.732237 3.675391 



level 4 -- 20 planes from 2.00 to 7.00 mm, 6 means and 21 covariance entries each, for 5,286 configurations (group values: all, cohort)
  the 6 coordinates are (zeta, y, z, ux, uy, uz): zeta along the bunch in um, y and z across it in um, and the three momenta

  the covariance at plane 3.00 mm, as the 6x6 matrix it is:
[[ 1.497e+01 -2.344e+00  2.061e+00 -3.867e+01 -4.046e-02  3.243e-01]
 [-2.344e+00  8.005e+02  2.083e+01 -4.209e+01  7.451e+01  1.431e+00]
 [ 2.061e+00  2.083e+01  3.041e+02  2.136e+01  1.431e+00  3.075e+01]
 [-3.867e+01 -4.209e+01  2.136e+01  9.661e+03  2.352e-01 -1.777e+00]
 [-4.046e-02  7.451e+01  1.431e+00  2.352e-01  1.314e+01  1.298e-01]
 [ 3.243e-01  1.431e+00  3.075e+01 -1.777e+00  1.298e-01  4.785e+00]]


### What is **not** in the pack, and it matters

- **The electrons themselves.** Level 4 is the bunch's mean and covariance, not its particles.
  The simulations do track individual electrons; those clouds are a separate download,
  level 5 (`level5/`, 5.3 GB for the 20 planes, 54.2 GB in full).
- **The laser and the plasma fields.** Nothing in the pack says what the laser was doing *inside*
  the cell — only what was set going in and what came out. The on-axis fields are a separate
  download too, level 6 (`level6/`, 1.6 GB).
- **The hidden true values**, and the 714 Campaign B and 2,446 Campaign A simulations behind
  them. See section 5.

## 2. Every file, opened

The loader above opens two of these. Here is all of them, with the real shape of each
rather than a description of it.

In [5]:
def describe(path):
    """One line on what a pack file actually holds, read from the file itself."""
    if path.endswith(".parquet"):
        f = pq.ParquetFile(path)
        return f"{f.metadata.num_rows:>9,} rows x {len(f.schema_arrow.names):>2} columns"
    if path.endswith(".npy"):
        return f"array of {np.load(path).shape[0]:,}"
    if path.endswith(".csv"):
        d = pd.read_csv(path, nrows=None)
        return f"{len(d):>9,} rows x {d.shape[1]:>2} columns"
    if path.endswith(".json"):
        d = json.load(open(path))
        if isinstance(d, dict):
            return "keys: " + ", ".join(list(d)[:6])
        return f"a list of {len(d)}"
    return ""

for name in sorted(os.listdir(PACK)):
    path = os.path.join(PACK, name)
    if not os.path.isfile(path):
        continue
    print(f"{name:38s} {os.path.getsize(path) / 1e6:7.1f} MB   {describe(path)}")

DATA_CARD.md                               0.0 MB   
campaign_A.parquet                        26.4 MB      13,073 rows x 35 columns
campaign_B.parquet                        94.5 MB       5,286 rows x 35 columns
campaign_B_moments.parquet                68.2 MB     211,360 rows x 57 columns
campaign_B_trajectories.parquet           11.2 MB     322,615 rows x 13 columns
common_energy_grid_MeV.npy                 0.0 MB   array of 200
pack_reference.json                        0.1 MB   keys: what, id_column, nothing_is_enforced, sections, scores
test_direct.parquet                        0.0 MB         357 rows x  6 columns
test_inverse.parquet                       0.2 MB       1,915 rows x 10 columns
test_inverse_B.parquet                     0.0 MB         357 rows x 10 columns
test_moments_planes.parquet                0.0 MB       3,213 rows x  7 columns
test_trajectory.parquet                    0.0 MB      40,341 rows x  7 columns
test_trajectory_ood.parquet                0.0 MB

### And what each one is for

| file | one sentence |
|---|---|
| `campaign_A.parquet` | Campaign A's finished bunches — four knobs in, the bunch out, one row per simulation. Holds **both** of Campaign A's scans; the `scan` column says which. |
| `campaign_B.parquet` | the same for Campaign B, with the density-profile knobs and the profile curve itself. |
| `campaign_B_trajectories.parquet` | the beam at 113 positions along the accelerator, for the Campaign B configurations that have one (count printed above). |
| `campaign_B_moments.parquet` | the bunch's six means and full 6×6 covariance at 20 planes. |
| `common_energy_grid_MeV.npy` | the one energy axis every `spec_common` column is on, 200 bins. |
| `pack_reference.json` | **one file, three sections**, keyed on `config` (the `plane_holdout` section lists planes, in mm): `reference_split` (the frozen split the internal reference models were trained on; its test half is the hidden test), `ood_split` (train below a density threshold; the organisers score you above it), `plane_holdout` (train on 11 planes, predict 9). Its `scores` hold every reference number. |
| `test_*.parquet` | the six hidden-test input files. **No target column appears in any of them.** |

The `test_*.parquet` files are what a submission is built from: inputs only, one row per prediction
the organisers will score.

## 3. How the files join, and the two traps in doing it

**`config` is one simulation, and it is the only id.** One row of `campaign_A.parquet`
or `campaign_B.parquet` is one simulation, `config` names it, it is unique in both files, and
all six test files join on it. So there is nothing to group and nothing to be careful about:
a split on `config` is a split on rows.

**Trap 1 — the moments table spells the id differently.** It is the string `"Config_1001"` in
`sim_id`; everywhere else, `test_moments_planes.parquet` included, the same id is the integer
`1001`. Strip the prefix before joining.

**Trap 2 — in the moments columns, `z` is a transverse coordinate.** The position along the
accelerator there is `plane_mm`. In the trajectory table the position along the accelerator is
`z_mm`. Two files, two meanings of the letter z.

In [6]:
print("campaign_A -- one row is one simulation, and config is unique in both scans:")
print(A.groupby("scan").agg(rows=("config", "size"),
                            configs=("config", "nunique"),
                            lowest=("config", "min"),
                            highest=("config", "max")).to_string(), "\n")

print("trap 1 -- strip the prefix before joining:")
print(f"  sim_id {mom.sim_id.iloc[0]!r}  ->  config {int(mom.config.iloc[0])}\n")

print("trap 2 -- plane_mm is the position along the accelerator; the z in these column")
print("          names is across the bunch. The knobs are already in this table, so join")
print("          it to the endpoint table when you want the finished bunch beside it:")
joined = m.merge(scal[["config", "E_med_MeV", "i_peak"]], on="config")
print(joined[["plane_mm", "mean_zeta_um", "mean_z_um", "mean_uz", "i_peak"]]
      .head(3).to_string(index=False))
print(f"\n  {joined.shape[0]} planes x {joined.shape[1]} columns; "
      f"the knobs {', '.join(B_KNOBS)} were already there")

campaign_A -- one row is one simulation, and config is unique in both scans:
         rows  configs  lowest  highest
scan                                   
random   9285     9285       5    12005
uniform  3788     3788  100000   103999 

trap 1 -- strip the prefix before joining:
  sim_id 'Config_0'  ->  config 0

trap 2 -- plane_mm is the position along the accelerator; the z in these column
          names is across the bunch. The knobs are already in this table, so join
          it to the endpoint table when you want the finished bunch beside it:
 plane_mm  mean_zeta_um  mean_z_um   mean_uz      i_peak
      2.0    -15.456481  -0.032117 -0.022541 5834.839324
      2.1    -15.778567  -0.052429 -0.028796 5834.839324
      2.2    -16.193909  -0.068532 -0.013705 5834.839324

  20 planes x 60 columns; the knobs P_max, cN2_max, L_inj, dip_frac, x_of were already there


### One injection flag, and the charge unit

`injected` is `q_pC >= 3.0`, one rule computed identically for both campaigns. The source
files' own labels are not shipped, so there is only ever one definition in play.

**`q_end` is in coulombs**, so a 3 pC cut is `3e-12`, not `3`; `q_pC` beside it is the same
number in picocoulombs.


## 4. The splits — how the reference numbers were made

**Nothing here is enforced.** Nobody checks what you trained on, and you cannot train on the
hidden test in any case, because its answers are not in the pack. `pack_reference.json` is a
record of how our internal reference numbers were produced, so that your number can be put
beside them.

That matters most for the questions that are *about* what you trained on. "Does accuracy still
climb when you halve the training data" is answered by running your own model at several sizes,
on subsets of the pool that you choose yourself — no fixed lists ship, and how you pick them is
part of the question. "How far
does it extrapolate" is answered by holding out high-density configurations — and those are
public, with their answers, so if you train on them the number measures nothing. Draw your own
line or reuse ours; the only thing you lose by ignoring it is comparability.

The file states its own purpose in a `what` field; here it is, read from the file.

In [7]:
for name in sorted(n for n in os.listdir(PACK) if n.endswith(".json")):
    d = json.load(open(os.path.join(PACK, name)))
    if isinstance(d, dict) and "what" in d:
        print(f"{name}\n    {d['what']}\n")

pack_reference.json
    everything you need to make your number comparable to the reference ones: the id lists each axis is defined on, and the scores themselves



**Which to use.** If you want a number you can compare with our internal references, train
on the `reference_split` section's train ids (all of them are public) and read the references as
a target — you cannot reproduce them yourself, because they are scored on the hidden test, whose
answers are not in the pack; the organisers score your submission on the same hidden
configurations. For a number you can compute yourself today, hold part of the public data out as
your own validation set: no fold ships, and how you split is up to you.

## 5. What is not here, and why

**714 Campaign B and 2,446 Campaign A simulations are absent.** They are the hidden test set. In Campaign A
that is the 1,915 simulations of `test_inverse.parquet` — 20 % of the random scan's runs that
produced a bunch — plus 531 untested runs that share a tested run's random draw and would give
its `a_0` away. The tested simulations are not in the public tables at all — one config, one
simulation, one row. A run that produced no bunch is nothing to test, so none of those is in
the test set.

**273 Campaign A runs were deleted, not flagged.** Their simulated laser envelope oscillates in
a way the physics does not allow, which means the run is a numerical failure rather than an
unusual machine setting. They are simply absent: no column, no list, nothing to filter. A
repaired envelope would have manufactured physics the simulation never produced.

**Three things were harmonised before you see them**, all described in `DATA_CARD.md`: the two
campaigns stored energy in different units, every spectrum was on its own energy axis, and the
two Campaign A scans had been post-processed under different injection rules. Energies are MeV
everywhere, spectra are on one shared grid, and `injected` is one rule — `q_pC >= 3.0` —
computed identically for everything.


## 6. The columns you will actually use

Units are what people get wrong here, so every column states one. **All energies are MeV**;
there is no gamma column and no unit flag to check.

### The ids

| column | meaning |
|---|---|
| `config` | **the simulation.** The only id: unique in both campaigns' files, and what every test file and every submission is keyed by |

`scan` (Campaign A only) is a label, not an id: which of the two scans a row came from, `random` or `uniform` on a grid.

### The knobs (what the operator sets)

| column | meaning | unit | campaign |
|---|---|---|---|
| `p_1` | injection-region pressure | mbar | A |
| `a_0` | laser amplitude (normalised vector potential) | dimensionless | A, B (fixed in B) |
| `c_N2` | nitrogen concentration | fraction | A |
| `x_of` | laser focal position | µm | A, B |
| `P_max` | peak pressure of the density profile | Pa | B |
| `cN2_max` | peak nitrogen fraction | fraction | B |
| `L_inj` | length of the injection region | mm | B |
| `dip_frac` | depth of the density dip | fraction | B |

### The beam (what comes out)

| column | meaning | unit |
|---|---|---|
| `E_med_MeV` | median beam energy | MeV |
| `E_mean_MeV` | mean beam energy | MeV |
| `dE_mad` | energy spread, median absolute deviation | **fraction of the energy**, not MeV |
| `dE_std` | energy spread, standard deviation | fraction |
| `q_end` | bunch charge | **coulombs** (so 3 pC is `3e-12`) |
| `i_peak` | peak current | A |
| `sigma_z`, `sigma_y` | bunch length and transverse size, RMS | m |
| `n_emit_x`, `n_emit_y` | normalised RMS emittance | m·rad |
| `div_rms` | RMS divergence | rad |
| `beta_x`, `alpha_x` | Twiss parameters | m, — |
| `spec`, `ener_axis_MeV` | the energy spectrum and its own axis | pC per MeV, MeV |
| `spec_common` | the same spectrum on the shared grid | pC per bin; sums to `q_pC` |
| `x_p`, `n_e_p` | the gas density profile | **µm**, m⁻³ |
| `envelope_validated` | Campaign A only: False above 73.888 mbar, where full-wave reruns did not confirm the simulation's laser model — trust those runs' energies least | — |

In [8]:
beam = ["E_med_MeV", "dE_mad", "q_end", "i_peak", "sigma_z", "n_emit_x", "div_rms"]
B.loc[B.injected, beam].describe().T[["min", "50%", "max"]]

,min,50%,max
E_med_MeV,6.697303e+01,1.339602e+02,2.412779e+02
dE_mad,1.902867e-02,7.511952e-02,5.084738e-01
q_end,3.037840e-12,6.846694e-11,3.006102e-10
i_peak,1.878321e+02,3.969007e+03,1.000096e+04
sigma_z,8.179158e-07,2.130239e-06,7.954796e-06
n_emit_x,1.809471e-07,5.277838e-07,9.159381e-06
div_rms,4.609011e-04,7.856679e-04,3.992739e-03


## 7. The spectra — one shared grid

Section 5 lists what the pack harmonised. For the spectra that means every row's `spec_common`
sits on the one 200-bin grid in `common_energy_grid_MeV.npy`, while its native spectrum keeps its
own axis in `ener_axis_MeV`.

`spec` is charge per MeV on the row's own axis; `spec_common` is the charge in pC in each
shared bin, so it sums to `q_pC`. The cell below checks that on one row.


In [9]:
print("the shared grid:", f"{grid[0]:.1f} to {grid[-1]:.1f} MeV in {len(grid)} bins")
spec = pd.read_parquet(f"{PACK}/campaign_B.parquet",
                       columns=["injected", "q_pC", "ener_axis_MeV", "spec_common"])
row = spec[spec.injected].iloc[0]
print("one row's native axis runs", f"{np.min(row.ener_axis_MeV):.1f} to "
      f"{np.max(row.ener_axis_MeV):.1f} MeV -- its own, not the shared one")
print(f"spec_common sums to {np.sum(row.spec_common):.2f} pC against q_pC = {row.q_pC:.2f} pC")


the shared grid: 25.8 to 510.3 MeV in 200 bins
one row's native axis runs 27.5 to 486.6 MeV -- its own, not the shared one
spec_common sums to 115.26 pC against q_pC = 115.26 pC


## 8. The task — the beam read back to the settings

$$(\texttt{E\_med\_MeV},\ \texttt{dE\_mad},\ \texttt{q\_end},\ \texttt{i\_peak},\
\texttt{n\_emit\_x},\ \texttt{sigma\_z},\ \texttt{sigma\_y},\ \texttt{div\_rms};\ \texttt{x\_of})
\;\longrightarrow\;(\texttt{p\_1},\ \texttt{a\_0},\ \texttt{c\_N2})$$

Campaign A's random scan, injected rows. Eight beam measurements, plus the focal position
`x_of`, which the operator sets and therefore knows — it is an input, not something to recover.

**Submit** a CSV with one row per test simulation: `config`, then your predicted `p_1`, `a_0`
and `c_N2`. Scored by R² per setting and their mean.

**The cell below is setup.** It defines the `train` and `test` frames every later cell uses.
A plain random split is enough: one row is one simulation, so nothing spans the two sides.


In [10]:
from sklearn.model_selection import train_test_split

# The task below is the random scan's: the hidden test set is drawn from it.
inj_a = A[(A.scan == "random") & A.injected]
train, test = train_test_split(inj_a, test_size=0.2, random_state=0)
print(f"train {len(train):,} rows / {train.config.nunique():,} configs"
      f"   test {len(test):,} rows / {test.config.nunique():,} configs")
print(f"configs in both sides: {len(set(train.config) & set(test.config))}")


train 5,766 rows / 5,766 configs   test 1,442 rows / 1,442 configs
configs in both sides: 0


In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Eight beam features plus the known focus in, three settings out -- the same inputs and
# targets as our reference ensemble, so the comparison below is like for like.
TARGETS = ["p_1", "a_0", "c_N2"]
FEATURES = ["E_med_MeV", "dE_mad", "q_end", "i_peak", "n_emit_x", "sigma_z", "sigma_y",
            "div_rms", "x_of"]

data_tr = train.dropna(subset=FEATURES + TARGETS)
data_te = test.dropna(subset=FEATURES + TARGETS)

LOG_COLUMNS = ["q_end", "i_peak", "n_emit_x", "sigma_y"]   # these span several decades


def inputs_with_logs(frame):
    """The model's input table: FEATURES, with LOG_COLUMNS on log10 (all > 0 on injected rows)."""
    features = frame[FEATURES].copy()
    for col in LOG_COLUMNS:
        features[col] = np.log10(features[col])
    return features.to_numpy()

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(inputs_with_logs(data_tr), data_tr[TARGETS].to_numpy())
# a linear model can leave the range it was trained on (a negative c_N2, say): keep it inside
TRAIN_MIN = data_tr[TARGETS].min().to_numpy()
TRAIN_MAX = data_tr[TARGETS].max().to_numpy()
pred = np.clip(model.predict(inputs_with_logs(data_te)), TRAIN_MIN, TRAIN_MAX)

dumb = DummyRegressor(strategy="mean").fit(
    inputs_with_logs(data_tr), data_tr[TARGETS].to_numpy()).predict(inputs_with_logs(data_te))

for i, t in enumerate(TARGETS):
    print(f"  {t:6s}  linear R2 {r2_score(data_te[t], pred[:, i]):+.4f}"
          f"   mean-predictor {r2_score(data_te[t], dumb[:, i]):+.4f}")
print(f"\n  mean linear R2 "
      f"{r2_score(data_te[TARGETS], pred, multioutput='uniform_average'):+.4f}")

  p_1     linear R2 +0.8609   mean-predictor -0.0003
  a_0     linear R2 +0.3897   mean-predictor -0.0000
  c_N2    linear R2 +0.8201   mean-predictor -0.0003

  mean linear R2 +0.6902


### Score it, then submit it

`pallas_score.py` beside this notebook is the scorer the organisers run. Score yourself on
your own held-out rows first — you have the answers for those. Then predict the **hidden test
set** (`test_inverse.parquet`, inputs only) and submit that file.

In [12]:
import sys
sys.path.insert(0, ".")
from pallas_score import score_submission

# Your own held-out rows, where you DO have the answers.
holdout_pred_inverse = pd.DataFrame(pred, columns=TARGETS)
holdout_pred_inverse.insert(0, "config", data_te.config.to_numpy())
holdout_truth_inverse = data_te[["config"] + TARGETS]
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in score_submission(holdout_pred_inverse, holdout_truth_inverse).items()})

{'p_1': 0.8609, 'a_0': 0.3897, 'c_N2': 0.8201, 'mean': 0.6902, 'n_rows': 1442}


In [13]:
# The hidden test set: inputs only, no targets. Predict and write.
test_inputs_inverse = pd.read_parquet(f"{PACK}/test_inverse.parquet")
test_pred_inverse = np.clip(model.predict(inputs_with_logs(test_inputs_inverse)), TRAIN_MIN, TRAIN_MAX)
submission = pd.DataFrame(test_pred_inverse, columns=TARGETS)
submission.insert(0, "config", test_inputs_inverse.config.to_numpy())
submission.to_csv("submission_inverse.csv", index=False)
print(f"wrote submission_inverse.csv  ({len(submission):,} rows)")
submission.head()

wrote submission_inverse.csv  (1,915 rows)


,config,p_1,a_0,c_N2
0,1,64.614553,1.263162,0.094039
1,1012,61.055340,1.399697,0.119997
2,1019,69.096857,1.397356,0.018446
3,1034,62.715534,1.313264,0.002173
4,1035,36.002672,1.291386,0.043804


### Only now: where that score sits

Try the task before reading this. Our mixture-density ensemble (in review, not yet published) reaches **mean R²
0.9100** and the **oracle ceiling is 0.9909** (both in `pack_reference.json`). The ceiling is
below 1 because the map is not injective — different settings give similar beams — so that gap
is degeneracy, not model error, and `a_0` is the hardest column for every model including ours.
A point estimate is therefore the wrong object; see "Inverse as a distribution" in `objectives.md`.

**Same task, same inputs.** The reference model reads the same eight beam features and the
known `x_of`, and predicts the same three settings, so your mean compares with it directly. One
difference is in your favour: the reference number was measured with realistic measurement noise
added to the beam features, and the test file here is noise-free. A clean input is an easier
problem, so a model as good as ours should score above 0.9100 here.

In [14]:
REFERENCE = {"p_1": 0.9563, "a_0": 0.8264, "c_N2": 0.9473}
ORACLE = {"p_1": 0.9940, "a_0": 0.9858, "c_N2": 0.9928}
holdout_scores_inverse = {t: r2_score(data_te[t], pred[:, i]) for i, t in enumerate(TARGETS)}
table = pd.DataFrame({"your baseline": holdout_scores_inverse,
                      "our ensemble (in review)": REFERENCE,
                      "oracle ceiling": ORACLE}).T
table["mean"] = table[TARGETS].mean(axis=1)
table.round(4)

,p_1,a_0,c_N2,mean
your baseline,0.8609,0.3897,0.8201,0.6902
our ensemble (in review),0.9563,0.8264,0.9473,0.9100
oracle ceiling,0.9940,0.9858,0.9928,0.9909


## 9. The other direction — the knobs read forward to the beam

$$(\texttt{P\_max},\ \texttt{cN2\_max},\ \texttt{L\_inj},\ \texttt{dip\_frac},\
\texttt{x\_of})\;\longrightarrow\;(\texttt{E\_med\_MeV},\ \texttt{dE\_mad},\
\texttt{q\_end})$$

Campaign B, injected rows. The same machine read forwards: the settings go in, the finished
bunch comes out. This is the direction our internal PALLAS forward model already does, so there
is a number to chase at the end of it.

**Submit** a CSV with one row per test simulation: `config`, then your predicted median energy,
energy spread and charge. Scored the same way, R² per column.


In [15]:
from pallas_score import normalised_to_physical, physical_to_normalised   # the scorer's own normalisation

DIRECT_INPUTS = ["P_max", "cN2_max", "L_inj", "dip_frac", "x_of"]
DIRECT_TARGETS = ["E_med_MeV", "dE_mad", "q_end"]

b_inj = B[B.injected]
b_train, b_test = train_test_split(b_inj, test_size=0.2, random_state=0)

# Fit on the scale the scorer uses, not on the raw numbers: charge spans two decades and
# the spread is a fraction, so a straight fit in physical units would chase the big bunches
# and ignore everything else. `physical_to_normalised` is the same function the organisers call.
def targets_to_normalised(frame, targets):
    return np.column_stack([physical_to_normalised(t, frame[t]) for t in targets])

def predictions_to_physical(pred, targets):
    """Back to the units a submission is written in -- the scorer re-applies the scales."""
    return pd.DataFrame({t: normalised_to_physical(t, pred[:, i]) for i, t in enumerate(targets)})

direct = make_pipeline(StandardScaler(), LinearRegression())
direct.fit(b_train[DIRECT_INPUTS].to_numpy(), targets_to_normalised(b_train, DIRECT_TARGETS))

holdout_pred_direct = predictions_to_physical(direct.predict(b_test[DIRECT_INPUTS].to_numpy()), DIRECT_TARGETS)
holdout_pred_direct.insert(0, "config", b_test.config.to_numpy())
print(f"train {len(b_train):,} rows   holdout {len(b_test):,} rows")
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in score_submission(holdout_pred_direct, b_test[["config"] + DIRECT_TARGETS]).items()})


train 2,305 rows   holdout 577 rows
{'E_med_MeV': 0.9399, 'dE_mad': 0.2614, 'q_end': 0.8138, 'mean': 0.6717, 'n_rows': 577}


In [16]:
# The hidden test set for this direction: 357 Campaign B settings, no answers.
test_inputs_direct = pd.read_parquet(f"{PACK}/test_direct.parquet")
submission_direct = predictions_to_physical(direct.predict(test_inputs_direct[DIRECT_INPUTS].to_numpy()), DIRECT_TARGETS)
submission_direct.insert(0, "config", test_inputs_direct.config.to_numpy())
submission_direct.to_csv("submission_direct.csv", index=False)
print(f"wrote submission_direct.csv  ({len(submission_direct):,} rows)")
submission_direct.head()


wrote submission_direct.csv  (357 rows)


,config,E_med_MeV,dE_mad,q_end
0,1041,124.330131,0.112563,7.262706e-11
1,1109,127.280267,0.124438,9.576904e-11
2,1088,129.730620,0.114757,8.084966e-11
3,103,157.227937,0.052572,7.860413e-11
4,1031,180.855683,0.075796,1.968004e-11


### And where that one sits

Our internal forward model (not published) reaches **0.9744** on the median energy, **0.8036** on the spread
and **0.9420** on the charge (`pack_reference.json`). It was trained on our own split of the
data rather than yours, so read the table below as a target to aim at, not as a like-for-like
race — the gap it shows is real and the energy spread is where it is widest.


In [17]:
DIRECT_REFERENCE = {"E_med_MeV": 0.9744, "dE_mad": 0.8036, "q_end": 0.9420}
holdout_scores_direct = score_submission(holdout_pred_direct, b_test[["config"] + DIRECT_TARGETS])
pd.DataFrame({"your baseline": {t: holdout_scores_direct[t] for t in DIRECT_TARGETS},
              "our internal forward model": DIRECT_REFERENCE}).T.round(4)


,E_med_MeV,dE_mad,q_end
your baseline,0.9399,0.2614,0.8138
our internal forward model,0.9744,0.8036,0.9420


## 10. The trajectory task — the file you hand in

Level 3 is one row per configuration **and** position `z_mm`, and so is the file you submit:
one row for every `(config, z_mm)` of `test_trajectory.parquet`, the six quantities in physical
units. No model is fitted here — that is the task (`objectives.md`).

To score yourself, hold out whole configurations of `campaign_B_trajectories.parquet` (never
single rows: all positions of one simulation go to the same side). The answer table is
`config, z_mm, L_inj` and the six targets — `L_inj` lets the scorer find each configuration's
plasma end, `L_inj + 3.2` mm — and nothing else, because every other column would count as a
target. `score_submission(your_predictions, answer_table)` then scores inside the plasma, the
ranked zone; `zone="drift"` prints the free-space drift after it, for information only.
`emit_um` and `div_mrad` are scored inside the plasma only.

In [18]:
TRAJECTORY_TARGETS = ["E_MeV", "q_pC", "dE_pct", "emit_um", "div_mrad", "sigz_um"]

test_inputs_trajectory = pd.read_parquet(f"{PACK}/test_trajectory.parquet")
print("the test file gives you:", ", ".join(test_inputs_trajectory.columns))

# the file you hand in: these two key columns, then one column per target
submission_trajectory = test_inputs_trajectory[["config", "z_mm"]].copy()
for target in TRAJECTORY_TARGETS:
    submission_trajectory[target] = np.nan          # replace with your model's predictions
print(f"{len(submission_trajectory):,} rows to fill -> submission_trajectory.csv")
submission_trajectory.head()

the test file gives you: config, z_mm, P_max, cN2_max, L_inj, dip_frac, x_of
40,341 rows to fill -> submission_trajectory.csv


,config,z_mm,E_MeV,q_pC,dE_pct,emit_um,div_mrad,sigz_um
0,1002,2.015055,NaN,NaN,NaN,NaN,NaN,NaN
1,1002,2.060053,NaN,NaN,NaN,NaN,NaN,NaN
2,1002,2.105055,NaN,NaN,NaN,NaN,NaN,NaN
3,1002,2.150053,NaN,NaN,NaN,NaN,NaN,NaN
4,1002,2.195155,NaN,NaN,NaN,NaN,NaN,NaN


## Where to go next

- **Beat both baselines** — anything non-linear will — then look at *where* each one fails.
- **Report a distribution, not a point**, and score its calibration. Different settings can
  produce nearly the same bunch, which is why even a perfect point predictor stops short on
  the inverse task.
- **Keep splitting on `config`**, so your fold stays comparable with the pack's reference split.
- **`objectives.md`** lists the easy, medium and hard objectives. The two tasks you just ran are
  among the easy ones; the scored trajectory task, the data-budget curve and the two
  out-of-distribution axes are where the challenge actually is.
- **`data/README.md`** gives the splits, the zone rule and the scoring conventions in full.
